# Transformers

Pytorch documentation [here](https://www.pytorch.org/) \
Keras documentation [here](https://keras.io/)

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"  # Set before importing keras

import numpy as np
import matplotlib.pyplot as plt

import keras
from keras import layers
from keras.datasets import imdb
from keras.utils import pad_sequences
from keras.optimizers import Adam

## Load data - IMDB

In [ ]:
# ------------------------------------------------------------
# 1) Load IMDB dataset
# ------------------------------------------------------------
vocab_size = 25000
max_len = 120

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

# Pad/truncate reviews to fixed length
X_train = pad_sequences(X_train, maxlen=max_len, padding="post", truncating="post")
X_test = pad_sequences(X_test, maxlen=max_len, padding="post", truncating="post")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


In [ ]:
# ------------------------------------------------------------
# 2) Positional Embedding Layer
# ------------------------------------------------------------
class PositionalEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embedding = layers.Embedding(vocab_size, embed_dim)
        self.position_embedding = layers.Embedding(max_len, embed_dim)

    def call(self, inputs):
        seq_len = keras.ops.shape(inputs)[-1]
        positions = keras.ops.arange(0, seq_len, 1)

        token_emb = self.token_embedding(inputs)
        pos_emb = self.position_embedding(positions)

        return token_emb + pos_emb


# ------------------------------------------------------------
# 3) Transformer Encoder Block
# ------------------------------------------------------------
class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training=False):
        # Self-attention
        attention_output = self.attention(inputs, inputs)
        attention_output = self.dropout1(attention_output, training=training)

        # First residual connection
        x = self.norm1(inputs + attention_output)

        # Feed-forward network
        ffn_output = self.ffn(x)
        ffn_output = self.dropout2(ffn_output, training=training)

        # Second residual connection
        return self.norm2(x + ffn_output)

## Create a graph model

In [ ]:
# ------------------------------------------------------------
# 4) Build Sequential Transformer Model
# ------------------------------------------------------------
embed_dim = 64
num_heads = 4
ff_dim = 128
dropout_rate = 0.2

model = keras.Sequential([
    layers.Input(shape=(max_len,), dtype="int32"),

    PositionalEmbedding(
        max_len=max_len,
        vocab_size=vocab_size,
        embed_dim=embed_dim
    ),

    TransformerEncoderBlock(
        embed_dim=embed_dim,
        num_heads=num_heads,
        ff_dim=ff_dim,
        dropout=dropout_rate
    ),

    layers.GlobalAveragePooling1D(),
    layers.Dropout(dropout_rate),
    layers.Dense(64, activation="relu"),
    layers.Dropout(dropout_rate),

    # Binary classification: positive / negative
    layers.Dense(1, activation="sigmoid")
])



In [ ]:
model.summary()

## Define loss function and optimizer

In [ ]:
# ------------------------------------------------------------
# 5) Compile
# ------------------------------------------------------------
model.compile(loss="binary_crossentropy", optimizer=Adam(learning_rate=1e-4), metrics=["accuracy"])

## Train model

In [ ]:

# ------------------------------------------------------------
# 6) Train
# ------------------------------------------------------------
history = model.fit( X_train, y_train,
                     epochs=5, batch_size=32,
                     validation_split=0.2,)


## Plot results

In [ ]:
# ================================
# 🔟 Plot progress and results
# ================================

_, axes = plt.subplots(1,2, figsize=(10,5))
axes[0].plot(history.history['loss'], c='b')
axes[0].plot(history.history['val_loss'], c='r')
axes[1].plot(history.history['accuracy'], c='b')
axes[1].plot(history.history['val_accuracy'], c='r')
axes[0].set_title("Loss"), axes[1].set_title('Accuracy')

## Compute metrics over ```X_test``` images

In [ ]:
# ------------------------------------------------------------
# 7) Evaluate
# ------------------------------------------------------------
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy:.4f}")